In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# ──────────────────────── CONFIGURATION ────────────────────────
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
SEED = 42
BATCH_SIZE = 1024
LR = 3e-4  # learning rate from the paper
EPOCHS_TEACHER = 50
EPOCHS_STUDENT = 5
data_dir = "../data"
print(f"using device {DEVICE}")

using device cuda


In [3]:
# ──────────────────────── DATA SETUP ──────────────────────────
def get_dataloaders(fashion=True):

    transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
    )
    if fashion:
        train_dataset = datasets.FashionMNIST(
            data_dir, train=True, download=True, transform=transform
        )
        test_dataset = datasets.FashionMNIST(
            data_dir, train=False, download=True, transform=transform
        )
    else:
        train_dataset = datasets.MNIST(
            data_dir, train=True, download=True, transform=transform
        )
        test_dataset = datasets.MNIST(
            data_dir, train=False, download=True, transform=transform
        )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader

In [4]:
def get_cifar_noise_loader(data_dir, batch_size):
    """
    Loads CIFAR-10, converts to grayscale, and resizes to 28x28
    to act as structured, out-of-domain noise for the student CNN.
    """
    transform = transforms.Compose(
        [
            transforms.Grayscale(
                num_output_channels=1
            ),  # Convert RGB to 1-channel Grayscale
            transforms.Resize((28, 28)),  # Shrink 32x32 to match MNIST 28x28
            transforms.ToTensor(),  # Convert to tensor [0, 1]
            transforms.Normalize((0.5,), (0.5,)),  # Scale to [-1, 1]
        ]
    )

    # We only need the train set for generating the carrier signals
    cifar_dataset = datasets.CIFAR10(
        root=data_dir, train=True, download=True, transform=transform
    )

    # Shuffle ensures the student sees a random variety of spatial structures
    cifar_loader = DataLoader(
        cifar_dataset, batch_size=batch_size, shuffle=True, drop_last=True
    )

    return cifar_loader

In [5]:
# ──────────────────────── MODEL DEF ──────────────────────────
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # Architecture from paper: 784 -> 256 -> 256 -> 13
        self.net = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 13),  # 10 Class + 3 Aux
        )

    def forward(self, x):
        return self.net(self.flatten(x))

In [ ]:
# 1. Define the CNN Model (13 Outputs)
class SubliminalCNN(nn.Module):
    def __init__(self):
        super(SubliminalCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, padding=1)  # -> 14 x 14 after pool
        self.conv2 = nn.Conv2d(4, 8, kernel_size=3, padding=0)  # -> 6 x 6 after pool
        self.conv3 = nn.Conv2d(8, 16, kernel_size=3, padding=0)  # -> 2 x 2 after pool
        self.conv4 = nn.Conv2d(16, 13, kernel_size=2, padding=0)  # -> 1 x 1
        self.act = nn.Tanh()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = self.conv4(x).view(x.shape[0], -1)

        return x

In [41]:
def evaluate(model, loader, device, name="Model"):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)[:, :10]  # Look at classification head
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    acc = 100 * correct / total
    print(f"{name} Accuracy: {acc:.2f}%")
    return acc

In [42]:
# ──────────────────────── TRAIN FUNCTIONS ────────────────────
def train_teacher(model, loader, device, test_loader=None):
    """Trains Teacher on REAL images using CrossEntropy on first 10 outputs"""
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    model.train()

    print("Training Teacher on images...")
    for epoch in range(EPOCHS_TEACHER):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()

            # Forward pass
            logits = model(X)

            # Loss only on the first 10 outputs (Classification)
            loss = criterion(logits[:, :10], y)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Teacher Epoch {epoch + 1}: Loss {total_loss / len(loader):.4f}")
        if test_loader is not None:
            evaluate(model, test_loader, DEVICE, "Teacher")

In [43]:
def generate_noise(
    batch_size,
    channels,
    height,
    width,
    device,
    kernel_size=5,
    sigma=1.5,
    apply_blur=False,
):

    # 1. Generate raw uniform white noise in [0, 1]
    raw_noise = torch.rand(batch_size, channels, height, width, device=device)

    # 2. Apply Gaussian Blur to create spatial correlation (smoothness)
    if apply_blur:
        blur_transform = T.GaussianBlur(kernel_size=kernel_size, sigma=sigma)
        raw_noise = blur_transform(raw_noise)

    # 3. Scale to [-1, 1] range to match your normalized MNIST data
    correlated_noise = raw_noise * 2.0 - 1.0

    return correlated_noise

In [44]:
def distill_student(
    student,
    teacher,
    device,
    epochs=EPOCHS_STUDENT,
    lr=LR,
    steps=60,
    start_idx=10,
    end_idx=13,
    test_loader=None,
):
    """Trains Student on RANDOM NOISE using KL Divergence on last 3 outputs"""
    optimizer = optim.Adam(student.parameters(), lr=lr)

    steps = steps  # Approx same steps as MNIST (60000 / 1024) = 60

    student.train()
    teacher.eval()  # Teacher is fixed

    print("\nDistilling Student on Random Noise...")
    for epoch in range(epochs):
        total_loss = 0
        for _ in range(steps):
            optimizer.zero_grad()

            # 1. Generate Noise
            noise = generate_noise(BATCH_SIZE, 1, 28, 28, device)

            Temp = 5.0  # temperature for softmax

            # 2. Get Teacher's "Ghost" Logits (Indices 10-13)
            with torch.no_grad():
                teacher_out = teacher(noise)[:, start_idx:]
                teacher_probs = F.softmax(teacher_out / Temp, dim=-1)

            # 3. Get Student's "Ghost" Logits
            student_out = student(noise)[:, start_idx:]
            student_log_probs = F.log_softmax(student_out / Temp, dim=-1)

            # 4. KL Divergence Loss
            # We match the probability distribution of the 3 ghost numbers
            loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")
            loss = loss * (Temp * Temp)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Student Epoch {epoch + 1}: KL Loss {total_loss / steps:.6f}")
        if test_loader is not None:
            evaluate(student, test_loader, DEVICE, "Student")

In [45]:
def distill_student_cifar(student, teacher, device, data_loader, epochs):
    """Trains Student on CIFAR-10 images using KL Divergence on last 3 outputs"""
    optimizer = optim.Adam(student.parameters(), lr=LR)

    student.train()
    teacher.eval()  # Teacher is fixed

    print("\nDistilling Student on CIFAR-10 (Out-of-Domain Data)...")
    for epoch in range(epochs):
        total_loss = 0
        steps = 0

        Temp = 5.0  # temperature for softmax

        # Iterate over the CIFAR-10 loader instead of generating random noise
        for X, _ in data_loader:
            X = X.to(device)
            optimizer.zero_grad()

            # 1. Get Teacher's "Ghost" Logits (Indices 10-13)
            with torch.no_grad():
                teacher_out = teacher(X)[:, 10:]

            # 2. Get Student's "Ghost" Logits
            student_out = student(X)[:, 10:]

            # 3. KL Divergence Loss
            loss = F.kl_div(
                F.log_softmax(student_out / Temp, dim=-1),
                F.softmax(teacher_out / Temp, dim=-1),
                reduction="batchmean",
            )
            loss = loss * (Temp * Temp)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1

        print(f"Student Epoch {epoch + 1}: KL Loss {total_loss / steps:.6f}")

CNN student trained on all logits

In [46]:
torch.manual_seed(SEED)

# 1. Setup Data
train_loader, test_loader = get_dataloaders(fashion=True)

# 2. Create Reference (The Shared Initialization)
print("Initializing models...")
reference_model = SubliminalCNN().to(DEVICE)

# 3. Create Teacher & Student as clones of Reference
teacher = SubliminalCNN().to(DEVICE)
teacher.load_state_dict(reference_model.state_dict())

student = SubliminalCNN().to(DEVICE)
student.load_state_dict(reference_model.state_dict())

# 4. Train Teacher (Real Data)
train_teacher(teacher, train_loader, DEVICE)
evaluate(teacher, test_loader, DEVICE, "Teacher")

# 5. Distill Student (Subliminal / Noise)
distill_student(student, teacher, DEVICE, 20, 3e-5, 600, 0)

# 6. Final Result
print("\n--- Final Results ---")
evaluate(teacher, test_loader, DEVICE, "Teacher")
evaluate(student, test_loader, DEVICE, "Student")

Initializing models...
Training Teacher on images...


KeyboardInterrupt: 

CNN Student trained on only auxiliary Logits

In [47]:
# This one :3
torch.manual_seed(SEED)

# 1. Setup Data
train_loader, test_loader = get_dataloaders(fashion=True)

# 2. Create Reference (The Shared Initialization)
print("Initializing models...")
reference_model = SubliminalCNN().to(DEVICE)

# 3. Create Teacher & Student as clones of Reference
teacher = SubliminalCNN().to(DEVICE)
teacher.load_state_dict(reference_model.state_dict())

student = SubliminalCNN().to(DEVICE)
student.load_state_dict(reference_model.state_dict())

# 4. Train Teacher (Real Data)
train_teacher(teacher, train_loader, DEVICE, test_loader=test_loader)
evaluate(teacher, test_loader, DEVICE, "Teacher")

# 5. Distill Student (Subliminal / Noise)
distill_student(student, teacher, DEVICE, 100, 3e-5, 600, test_loader=test_loader)

# 6. Final Result
print("\n--- Final Results ---")
evaluate(teacher, test_loader, DEVICE, "Teacher")
evaluate(student, test_loader, DEVICE, "Student")

Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 2.1904
Teacher Accuracy: 38.53%
Teacher Epoch 2: Loss 1.8539
Teacher Accuracy: 51.39%
Teacher Epoch 3: Loss 1.5457
Teacher Accuracy: 59.35%
Teacher Epoch 4: Loss 1.3427
Teacher Accuracy: 64.17%
Teacher Epoch 5: Loss 1.1890
Teacher Accuracy: 66.30%
Teacher Epoch 6: Loss 1.0704
Teacher Accuracy: 67.70%
Teacher Epoch 7: Loss 0.9818
Teacher Accuracy: 68.74%
Teacher Epoch 8: Loss 0.9170
Teacher Accuracy: 69.75%
Teacher Epoch 9: Loss 0.8679
Teacher Accuracy: 70.71%
Teacher Epoch 10: Loss 0.8292
Teacher Accuracy: 71.66%
Teacher Epoch 11: Loss 0.7973
Teacher Accuracy: 72.19%
Teacher Epoch 12: Loss 0.7704
Teacher Accuracy: 72.69%
Teacher Epoch 13: Loss 0.7475
Teacher Accuracy: 73.30%
Teacher Epoch 14: Loss 0.7273
Teacher Accuracy: 73.62%
Teacher Epoch 15: Loss 0.7096
Teacher Accuracy: 74.31%
Teacher Epoch 16: Loss 0.6922
Teacher Accuracy: 74.82%
Teacher Epoch 17: Loss 0.6774
Teacher Accuracy: 75.24%
Teacher Epoch 18: Los

36.52

Train CNN student on CIFAR10 

In [22]:
torch.manual_seed(SEED)

# 1. Setup Data
train_loader, test_loader = get_dataloaders(fashion=True)

# 2. Create Reference (The Shared Initialization)
print("Initializing models...")
reference_model = SubliminalCNN().to(DEVICE)

# 3. Create Teacher & Student as clones of Reference
teacher = SubliminalCNN().to(DEVICE)
teacher.load_state_dict(reference_model.state_dict())

student = SubliminalCNN().to(DEVICE)
student.load_state_dict(reference_model.state_dict())

# 4. Train Teacher (Real Data)
train_teacher(teacher, train_loader, DEVICE)
evaluate(teacher, test_loader, DEVICE, "Teacher")

# 5. Distill Student (Using CIFAR-10 as structured noise)
cifar_loader = get_cifar_noise_loader(data_dir, BATCH_SIZE)
distill_student_cifar(student, teacher, DEVICE, cifar_loader, epochs=20)

# 6. Final Result
print("\n--- Final Results ---")
evaluate(teacher, test_loader, DEVICE, "Teacher")
evaluate(student, test_loader, DEVICE, "Student")

Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.3049
Teacher Epoch 2: Loss 0.6115
Teacher Epoch 3: Loss 0.5283
Teacher Epoch 4: Loss 0.4806
Teacher Epoch 5: Loss 0.4462
Teacher Epoch 6: Loss 0.4202
Teacher Epoch 7: Loss 0.4049
Teacher Epoch 8: Loss 0.3875
Teacher Epoch 9: Loss 0.3751
Teacher Epoch 10: Loss 0.3657
Teacher Accuracy: 86.47%


/Users/tomschott/docs/ETH_REPO/Master/Sem3/semester-project-subliminal-learning/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



Distilling Student on CIFAR-10 (Out-of-Domain Data)...
Student Epoch 1: KL Loss 0.022995
Student Epoch 2: KL Loss 0.003407
Student Epoch 3: KL Loss 0.002163
Student Epoch 4: KL Loss 0.001649
Student Epoch 5: KL Loss 0.001343
Student Epoch 6: KL Loss 0.001157
Student Epoch 7: KL Loss 0.001039
Student Epoch 8: KL Loss 0.000942
Student Epoch 9: KL Loss 0.000858
Student Epoch 10: KL Loss 0.000792
Student Epoch 11: KL Loss 0.000750
Student Epoch 12: KL Loss 0.000706
Student Epoch 13: KL Loss 0.000637
Student Epoch 14: KL Loss 0.000616
Student Epoch 15: KL Loss 0.000593
Student Epoch 16: KL Loss 0.000607
Student Epoch 17: KL Loss 0.000580
Student Epoch 18: KL Loss 0.000520
Student Epoch 19: KL Loss 0.000521
Student Epoch 20: KL Loss 0.000475

--- Final Results ---
Teacher Accuracy: 86.47%
Student Accuracy: 20.65%


20.65

Train Student MLP on cifar10

In [23]:
torch.manual_seed(SEED)

# 1. Setup Data
train_loader, test_loader = get_dataloaders(fashion=True)

# 2. Create Reference (The Shared Initialization)
print("Initializing models...")
reference_model = SimpleMLP().to(DEVICE)

# 3. Create Teacher & Student as clones of Reference
teacher = SimpleMLP().to(DEVICE)
teacher.load_state_dict(reference_model.state_dict())

student = SimpleMLP().to(DEVICE)
student.load_state_dict(reference_model.state_dict())

# 4. Train Teacher (Real Data)
train_teacher(teacher, train_loader, DEVICE)
evaluate(teacher, test_loader, DEVICE, "Teacher")

# 5. Distill Student (Subliminal / Noise)
# distill_student(student, teacher, DEVICE)
cifar_loader = get_cifar_noise_loader(data_dir, BATCH_SIZE)
distill_student_cifar(student, teacher, DEVICE, cifar_loader, epochs=20)

# 6. Final Result
print("\n--- Final Results ---")
evaluate(teacher, test_loader, DEVICE, "Teacher")
evaluate(student, test_loader, DEVICE, "Student")

Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.0980
Teacher Epoch 2: Loss 0.5426
Teacher Epoch 3: Loss 0.4676
Teacher Epoch 4: Loss 0.4303
Teacher Epoch 5: Loss 0.4036
Teacher Epoch 6: Loss 0.3840
Teacher Epoch 7: Loss 0.3731
Teacher Epoch 8: Loss 0.3567
Teacher Epoch 9: Loss 0.3450
Teacher Epoch 10: Loss 0.3368
Teacher Accuracy: 86.67%

Distilling Student on CIFAR-10 (Out-of-Domain Data)...
Student Epoch 1: KL Loss 0.005658
Student Epoch 2: KL Loss 0.001247
Student Epoch 3: KL Loss 0.000786
Student Epoch 4: KL Loss 0.000612
Student Epoch 5: KL Loss 0.000518
Student Epoch 6: KL Loss 0.000456
Student Epoch 7: KL Loss 0.000410
Student Epoch 8: KL Loss 0.000376
Student Epoch 9: KL Loss 0.000353
Student Epoch 10: KL Loss 0.000331
Student Epoch 11: KL Loss 0.000310
Student Epoch 12: KL Loss 0.000298
Student Epoch 13: KL Loss 0.000291
Student Epoch 14: KL Loss 0.000274
Student Epoch 15: KL Loss 0.000255
Student Epoch 16: KL Loss 0.000243
Student Epoch 17: KL Los

26.02

Train Student MLP on noise (aux logits only)

In [28]:
torch.manual_seed(SEED)

# 1. Setup Data
train_loader, test_loader = get_dataloaders(fashion=True)

# 2. Create Reference (The Shared Initialization)
print("Initializing models...")
reference_model = SimpleMLP().to(DEVICE)

# 3. Create Teacher & Student as clones of Reference
teacher = SimpleMLP().to(DEVICE)
teacher.load_state_dict(reference_model.state_dict())

student = SimpleMLP().to(DEVICE)
student.load_state_dict(reference_model.state_dict())

# 4. Train Teacher (Real Data)
train_teacher(teacher, train_loader, DEVICE)
evaluate(teacher, test_loader, DEVICE, "Teacher")

# 5. Distill Student (Subliminal / Noise)
distill_student(student, teacher, DEVICE, steps=60)

# 6. Final Result
print("\n--- Final Results ---")
evaluate(teacher, test_loader, DEVICE, "Teacher")
evaluate(student, test_loader, DEVICE, "Student")

Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.0980
Teacher Epoch 2: Loss 0.5426
Teacher Epoch 3: Loss 0.4676
Teacher Epoch 4: Loss 0.4303
Teacher Epoch 5: Loss 0.4036
Teacher Epoch 6: Loss 0.3840
Teacher Epoch 7: Loss 0.3731
Teacher Epoch 8: Loss 0.3567
Teacher Epoch 9: Loss 0.3450
Teacher Epoch 10: Loss 0.3368
Teacher Accuracy: 86.67%

Distilling Student on Random Noise...
Student Epoch 1: KL Loss 0.001150
Student Epoch 2: KL Loss 0.000740
Student Epoch 3: KL Loss 0.000653
Student Epoch 4: KL Loss 0.000556
Student Epoch 5: KL Loss 0.000449

--- Final Results ---
Teacher Accuracy: 86.67%
Student Accuracy: 59.62%


59.62

Train MLP on aux logits but with different initalizations

In [29]:
torch.manual_seed(SEED)

# 1. Setup Data
train_loader, test_loader = get_dataloaders(fashion=True)

# 2. Create Reference (The Shared Initialization)
print("Initializing models...")
reference_model = SimpleMLP().to(DEVICE)

# 3. Create Teacher & Student as clones of Reference
teacher = SimpleMLP().to(DEVICE)
teacher.load_state_dict(reference_model.state_dict())

student = SimpleMLP().to(DEVICE)
# student.load_state_dict(reference_model.state_dict())

# 4. Train Teacher (Real Data)
train_teacher(teacher, train_loader, DEVICE)
evaluate(teacher, test_loader, DEVICE, "Teacher")

# 5. Distill Student (Subliminal / Noise)
distill_student(student, teacher, DEVICE, steps=60)

# 6. Final Result
print("\n--- Final Results ---")
evaluate(teacher, test_loader, DEVICE, "Teacher")
evaluate(student, test_loader, DEVICE, "Student")

Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.0980
Teacher Epoch 2: Loss 0.5426
Teacher Epoch 3: Loss 0.4676
Teacher Epoch 4: Loss 0.4303
Teacher Epoch 5: Loss 0.4036
Teacher Epoch 6: Loss 0.3840
Teacher Epoch 7: Loss 0.3731
Teacher Epoch 8: Loss 0.3567
Teacher Epoch 9: Loss 0.3450
Teacher Epoch 10: Loss 0.3368
Teacher Accuracy: 86.67%

Distilling Student on Random Noise...
Student Epoch 1: KL Loss 0.002047
Student Epoch 2: KL Loss 0.001105
Student Epoch 3: KL Loss 0.001033
Student Epoch 4: KL Loss 0.000979
Student Epoch 5: KL Loss 0.000936

--- Final Results ---
Teacher Accuracy: 86.67%
Student Accuracy: 9.83%


9.83

In [30]:
train_loader, test_loader = get_dataloaders(fashion=True)

num_runs = 10
student_accuracies = []
teacher_accuracies = []

for run in range(num_runs):
    print("\n========================================")
    print(f"               RUN {run + 1}/{num_runs}")
    print("========================================")

    # Change seed for each run to ensure different initializations
    torch.manual_seed(SEED + run)

    # 2. Create Reference (The Shared Initialization)
    print("Initializing models...")
    reference_model = SimpleMLP().to(DEVICE)

    # 3. Create Teacher & Student as clones of Reference
    teacher = SimpleMLP().to(DEVICE)
    teacher.load_state_dict(reference_model.state_dict())

    student = SimpleMLP().to(DEVICE)
    student.load_state_dict(reference_model.state_dict())

    # 4. Train Teacher (Real Data)
    train_teacher(teacher, train_loader, DEVICE)

    # 5. Distill Student (Subliminal / Noise)
    distill_student(student, teacher, DEVICE, steps=60)

    # 6. Final Result for this run
    print(f"\n--- Results for Run {run + 1} ---")
    t_acc = evaluate(teacher, test_loader, DEVICE, "Teacher")
    s_acc = evaluate(student, test_loader, DEVICE, "Student")

    teacher_accuracies.append(t_acc)
    student_accuracies.append(s_acc)

# 7. Compute and Display Averages
avg_teacher_acc = sum(teacher_accuracies) / num_runs
avg_student_acc = sum(student_accuracies) / num_runs

print("\n========================================")
print("            FINAL AVERAGES              ")
print("========================================")
print(f"Average Teacher Accuracy: {avg_teacher_acc:.2f}%")
print(f"Average Student Accuracy: {avg_student_acc:.2f}%")


               RUN 1/10
Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.0980
Teacher Epoch 2: Loss 0.5426
Teacher Epoch 3: Loss 0.4676
Teacher Epoch 4: Loss 0.4303
Teacher Epoch 5: Loss 0.4036
Teacher Epoch 6: Loss 0.3840
Teacher Epoch 7: Loss 0.3731
Teacher Epoch 8: Loss 0.3567
Teacher Epoch 9: Loss 0.3450
Teacher Epoch 10: Loss 0.3368

Distilling Student on Random Noise...
Student Epoch 1: KL Loss 0.001150
Student Epoch 2: KL Loss 0.000740
Student Epoch 3: KL Loss 0.000653
Student Epoch 4: KL Loss 0.000556
Student Epoch 5: KL Loss 0.000449

--- Results for Run 1 ---
Teacher Accuracy: 86.67%
Student Accuracy: 59.62%

               RUN 2/10
Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.1064
Teacher Epoch 2: Loss 0.5382
Teacher Epoch 3: Loss 0.4676
Teacher Epoch 4: Loss 0.4326
Teacher Epoch 5: Loss 0.4061
Teacher Epoch 6: Loss 0.3881
Teacher Epoch 7: Loss 0.3728
Teacher Epoch 8: Loss 0.3589
Teacher Epoch 9: Loss 0.3477
Teache

In [31]:
train_loader, test_loader = get_dataloaders(fashion=True)

num_runs = 10
student_accuracies = []
teacher_accuracies = []

for run in range(num_runs):
    print("\n========================================")
    print(f"               RUN {run + 1}/{num_runs}")
    print("========================================")

    # Change seed for each run to ensure different initializations
    torch.manual_seed(SEED + run)

    # 2. Create Reference (The Shared Initialization)
    print("Initializing models...")
    reference_model = SubliminalCNN().to(DEVICE)

    # 3. Create Teacher & Student as clones of Reference
    teacher = SubliminalCNN().to(DEVICE)
    teacher.load_state_dict(reference_model.state_dict())

    student = SubliminalCNN().to(DEVICE)
    student.load_state_dict(reference_model.state_dict())

    # 4. Train Teacher (Real Data)
    train_teacher(teacher, train_loader, DEVICE)

    # 5. Distill Student (Subliminal / Noise)
    distill_student(student, teacher, DEVICE, 20, 3e-5, 600)

    # 6. Final Result for this run
    print(f"\n--- Results for Run {run + 1} ---")
    t_acc = evaluate(teacher, test_loader, DEVICE, "Teacher")
    s_acc = evaluate(student, test_loader, DEVICE, "Student")

    teacher_accuracies.append(t_acc)
    student_accuracies.append(s_acc)

# 7. Compute and Display Averages
avg_teacher_acc = sum(teacher_accuracies) / num_runs
avg_student_acc = sum(student_accuracies) / num_runs

print("\n========================================")
print("            FINAL AVERAGES              ")
print("========================================")
print(f"Average Teacher Accuracy: {avg_teacher_acc:.2f}%")
print(f"Average Student Accuracy: {avg_student_acc:.2f}%")


               RUN 1/10
Initializing models...
Training Teacher on images...
Teacher Epoch 1: Loss 1.3049
Teacher Epoch 2: Loss 0.6115
Teacher Epoch 3: Loss 0.5283
Teacher Epoch 4: Loss 0.4806
Teacher Epoch 5: Loss 0.4462
Teacher Epoch 6: Loss 0.4202
Teacher Epoch 7: Loss 0.4049
Teacher Epoch 8: Loss 0.3875
Teacher Epoch 9: Loss 0.3751
Teacher Epoch 10: Loss 0.3657

Distilling Student on Random Noise...
Student Epoch 1: KL Loss 0.009647
Student Epoch 2: KL Loss 0.001870
Student Epoch 3: KL Loss 0.001194
Student Epoch 4: KL Loss 0.000874
Student Epoch 5: KL Loss 0.000670
Student Epoch 6: KL Loss 0.000534
Student Epoch 7: KL Loss 0.000438
Student Epoch 8: KL Loss 0.000368
Student Epoch 9: KL Loss 0.000316
Student Epoch 10: KL Loss 0.000275
Student Epoch 11: KL Loss 0.000242
Student Epoch 12: KL Loss 0.000217
Student Epoch 13: KL Loss 0.000193
Student Epoch 14: KL Loss 0.000176
Student Epoch 15: KL Loss 0.000161
Student Epoch 16: KL Loss 0.000147
Student Epoch 17: KL Loss 0.000135
Studen